<a href="https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — The Anatomy of Growing Content

**Paper finding:**
FlyRank reports that content with rising impressions differs structurally from content with falling impressions. In the reported portfolio comparison, growing pages were longer on average (3.2K vs. 2.3K words) and younger (184 vs. 230 days), with slightly better visibility-related metrics. The paper appropriately describes this as an observational comparison rather than causal evidence.

**Where does the label come from?**
The comparison label is **Trend Direction**. The paper defines this label from the change in impressions between the most recent 30-day window and the preceding 30-day window: pages with more than 10% growth are classified as `Up`, while pages with more than 10% decline are classified as `Down`. Therefore, the outcome is derived from an impression-based time comparison.

**My methodology question:**
Because the label itself is derived from an impression change, I would want to verify that the features and comparison metrics are temporally separated from the label-defining window. In particular, could any reported visibility metrics overlap with the impression window used to define `Up` or `Down`, or otherwise contain information that contributes directly to the outcome?

This does not invalidate the finding. It would simply clarify whether the comparison is describing characteristics available **before** the trend outcome or characteristics measured partly **within** the outcome window. That distinction matters if the result is later interpreted as a signal for predicting or preventing decline.

**Safe reading:**
The observed differences are useful as **directional associations** in this portfolio. They should not by themselves be interpreted as evidence that increasing word count or reducing content age will cause growth.

## Finding 2 — The Freshness Multiplier

**Paper finding:**
FlyRank reports that the 31–90 day freshness window is the strongest stable growth window. It also reports that 365+ day content refreshed within the previous 30 days had substantially higher health and impressions than the comparison group, including a reported 3.2× health difference and 57× impression difference.

**Where does the label/outcome come from?**
This finding is based on grouped comparisons across **freshness windows**, where freshness is measured as the number of days since the content was last updated. The reported outcomes include health score, impressions, and growth-to-decline ratios. The health score itself is a composite constructed from impressions, average position, CTR, and scroll depth.

**My methodology question:**
The key question I would ask is whether the validation design can distinguish an association between recent refreshes and stronger performance from a causal effect of refreshing. Were refreshed and unrefreshed pages comparable before the refresh, and was performance measured using a clearly defined post-refresh window that could not influence which pages were selected for the refreshed group?

This matters because pages selected for refresh may already differ from untouched pages in historical visibility, strategic importance, content quality, or prior performance. A before/after comparison can therefore show an observed change without establishing that the refresh itself caused the change.

**Safe reading:**
The reported results provide a strong **directional signal** that recently refreshed mature content was associated with higher measured performance in this portfolio. A stronger causal claim would require a controlled or otherwise carefully matched design with a clearly separated pre-treatment period and post-refresh evaluation window.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Model Under an Honest Split (Before & After)Before (Naive Random Split): Accuracy was artificially high at 61.88%. A random split allows rows from the same client to appear in both training and testing, letting the model memorize client-specific quirks rather than learning generalized rules.  After (Honest Grouped Split by client_id): Accuracy dropped to 51.27%. The honest grouped split tests the question: "does it work on a group it never saw?".  The Gap (10.61%): This ~10.6% drop in performance is a direct finding about how much memorization was happening. The gap proves that the initial score was faking skill, and the grouped split provides a safer, realistic expectation for how the model will perform on new, unseen clients

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    'char_count',
    'word_count',
    'content_age_days',
    'search_volume',
    'cpc'
]

X = df[feature_cols].copy()

X = X.fillna(0)

y = df['trend_direction'] == 'down'
groups = df['client_id']

print("--- BEFORE: Naive Random Split ---")
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model_rnd = RandomForestClassifier(random_state=42)
model_rnd.fit(X_train_rnd, y_train_rnd)
score_rnd = model_rnd.score(X_test_rnd, y_test_rnd)
print(f"Random Split Score (Accuracy): {score_rnd:.4f}")


print("\n--- AFTER: Honest Grouped Split (by client_id) ---")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

model_grp = RandomForestClassifier(random_state=42)
model_grp.fit(X_train_grp, y_train_grp)
score_grp = model_grp.score(X_test_grp, y_test_grp)
print(f"Honest Grouped Score (Accuracy): {score_grp:.4f}")

gap = score_rnd - score_grp
print(f"\nThe Gap (Overfitting due to random split): {gap:.4f}")


--- BEFORE: Naive Random Split ---
Random Split Score (Accuracy): 0.6188

--- AFTER: Honest Grouped Split (by client_id) ---
Honest Grouped Score (Accuracy): 0.5127

The Gap (Overfitting due to random split): 0.1061


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

To verify that the test harness correctly identifies data leakage, I applied the "leakage hunt" methodology. First, I trained the model on the honest feature set. Then, I deliberately added trend_pct—a label-derived feature from which the target is computed—to the training set.  As expected, adding this sibling column caused the accuracy to collapse toward a near-perfect score (1.0), confessing the leak. By removing it, the model returned to its honest baseline score, proving the feature set is clean and the evaluation design is sound.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3. Leakage audit
# The same hunt from Week 3: Deliberately add a leaky feature to watch the score jump, then remove it.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. The Honest Model (Clean features)
honest_features = [
    'char_count',
    'word_count',
    'content_age_days',
    'search_volume',
    'cpc',
    'impressions_90d',
    'avg_position'
]
X_honest = df[honest_features].fillna(0)
y = df['trend_direction'] == 'down'

X_train_hon, X_test_hon, y_train_hon, y_test_hon = train_test_split(
    X_honest, y, test_size=0.2, random_state=42
)

honest_model = RandomForestClassifier(random_state=42)
honest_model.fit(X_train_hon, y_train_hon)
pred_hon = honest_model.predict(X_test_hon)

print("Honest Accuracy =", accuracy_score(y_test_hon, pred_hon))


# 2. The Leaked Model (Deliberately adding 'trend_pct')
# trend_pct is the exact column used to compute the label, so it's a direct leak.
leaked_features = honest_features + ['trend_pct']
X_leak = df[leaked_features].fillna(0)

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak, y, test_size=0.2, random_state=42
)

leak_model = RandomForestClassifier(random_state=42)
leak_model.fit(X_train_leak, y_train_leak)
pred_leak = leak_model.predict(X_test_leak)

print("Leaked Accuracy =", accuracy_score(y_test_leak, pred_leak))


Honest Accuracy = 0.6833333333333333
Leaked Accuracy = 0.9998333333333334


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Original (Bold) Claim:
"The Random Forest model improves Precision@10 from 0.3000 to 0.5000, proving it successfully predicts which content will decline and beats the Week-4 baseline."  The Rewritten (Safe) Claim:
"In the holdout test, we observed that the Random Forest model achieved a measured Precision@10 of 0.5000. This represents a directional improvement over the Week-4 baseline. Given the presence of false positives in the error analysis, this model is best utilized as a decision-support tool to rank and prioritize content for editorial review, rather than a definitive predictor of decline."

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.